# GA4 신규 사용자 첫 세션 행동 → D7 구매 예측 모델링 (LightGBM + SHAP)

**목적** — 첫 세션 행동 중 이후 구매·재방문과 연결되는 행동(X)과 수행 기준(N)을 데이터로 확인

| 항목 | 내용 |
|---|---|
| 데이터 | `ga4_portfolio.modeling_mart` (`sql/03_modeling_mart.sql`로 생성) |
| 대상 | 첫 세션 미구매 · D7 관찰 가능 신규 사용자 236,648명 |
| 타깃 | `target_purchase_d7` 이후 D7 구매 (0.47%) / `target_return_d1_d7` D1~D7 재방문 (5.00%) |
| 모델 | Logistic Regression(기준) vs LightGBM |
| 평가 | PR-AUC, ROC-AUC, 상위 10% Lift · Capture, MCC |
| 해석 | SHAP 중요도, 값별 평균 SHAP(임계값 탐색) |

> 실행 환경: Google Colab 또는 BigQuery 인증이 된 로컬 환경 (`GCP_PROJECT_ID` 환경변수 설정)

In [1]:
# [Colab 전용] 드라이브 연결 · 패키지 설치 · 한글 폰트
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/데이터분석_포트폴리오/ga4-first-session-conversion/notebooks"
!pip install -q -r ../requirements.txt
!apt-get -qq install fonts-nanum > /dev/null

import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/데이터분석_포트폴리오/ga4-first-session-conversion/notebooks
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. 설정

In [2]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import shap

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             matthews_corrcoef, precision_score, recall_score)

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

PROJECT_ID  = os.environ.get("GCP_PROJECT_ID", "ga4-analysis-504708")
TABLE       = f"`{PROJECT_ID}.ga4_portfolio.modeling_mart`"   # sql/03_modeling_mart.sql 결과
SEED        = 42
TOP_PCT     = 0.10      # 상위 10% 타기팅 기준
SHAP_SAMPLE = 20000     # SHAP 계산 표본 수 (속도용)


TARGETS = {
    "purchase": "target_purchase_d7",
    "revisit":  "target_return_d1_d7",   # 24~168시간 재방문
}

# 개선 레버로 해석할 행동 변수
BEHAVIOR_FEATURES = [
    "view_search_results_count",
    "search_then_view_count",      # 검색 후 상품 조회 (1차 X 후보)
    "other_view_count",            # 검색 외 경로 상품 조회
    "select_item_count",
    "add_to_cart_count",
    "view_promotion_count",
    "select_promotion_count",
    "distinct_item_viewed",       # distinct_category_viewed와 상관 0.999 → 하나만 사용
]
# 결제 직전 행동 : 구매 의도 자체에 가까워 레버 해석 시 주의
# (add_shipping_info_count는 begin_checkout_count와 상관 1.0이라 제외)
CHECKOUT_FEATURES = [
    "begin_checkout_count",
    "add_payment_info_count"
]
# 참여 지표 : 결과에 가까운 지표라 레버보다는 통제 성격
ENGAGEMENT_FEATURES = ["engagement_sec", "session_duration_sec", "event_count"]
# 통제 변수 : 개선 레버로 해석하지 않음
CATEGORICAL_FEATURES = ["device_category", "traffic_medium"]

# 참여 지표는 상세 조회 횟수와 겹쳐 행동 변수의 기여를 흡수하므로 최종 모델에서는 제외
# (True로 바꿔 실행하면 참여 지표 포함 버전과 순위 비교 가능)
INCLUDE_ENGAGEMENT = False

# view_item_count 는 search_then_view + other_view 로 분해되어 제외
NUMERIC_FEATURES = (BEHAVIOR_FEATURES + CHECKOUT_FEATURES
                    + (ENGAGEMENT_FEATURES if INCLUDE_ENGAGEMENT else []))
OUT_DIR = "outputs_all" if INCLUDE_ENGAGEMENT else "outputs_behavior"
os.makedirs(OUT_DIR, exist_ok=True)
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

FEATURE_GROUP = {
    **{f: "행동(레버)" for f in BEHAVIOR_FEATURES},
    **{f: "결제 직전" for f in CHECKOUT_FEATURES},
    **{f: "참여(통제)" for f in ENGAGEMENT_FEATURES},
    **{f: "통제 변수" for f in CATEGORICAL_FEATURES},
    "view_promotion_count": "진입 경로(통제)",   # 메인 페이지 방문과 강하게 연결돼 레버로 해석하지 않음
}

## 2. 데이터 로드 및 점검

In [3]:
# 데이터 로드 · 전처리
# Colab에서 실행 시: from google.colab import auth; auth.authenticate_user()
def load_data():
    from google.cloud import bigquery
    client = bigquery.Client(project=PROJECT_ID)
    return client.query(f"SELECT * FROM {TABLE}").to_dataframe()


def prepare(df):
    df = df[df["first_session_purchase_flag"] == 0].copy()   # 첫 세션 구매자 제외
    for c in NUMERIC_FEATURES:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(float)
    for c in CATEGORICAL_FEATURES:
        df[c] = df[c].fillna("(missing)").astype(str).astype("category")
    for t in TARGETS.values():
        df[t] = df[t].astype(int)
    return df.sort_values("user_pseudo_id").reset_index(drop=True)

In [4]:
# 상관관계 점검 — |ρ| ≥ 0.8인 변수 쌍은 SHAP 기여도가 나뉘므로 묶어서 해석

def check_correlation(df, threshold=0.8):
    corr = df[NUMERIC_FEATURES].corr(method="spearman")
    pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
                 .stack().reset_index())
    pairs.columns = ["feature_1", "feature_2", "spearman"]
    pairs = pairs[pairs["spearman"].abs() >= threshold]
    return pairs.sort_values("spearman", ascending=False, key=abs)

## 3. 평가 · 학습 · SHAP 함수

In [5]:
# 평가 함수 — 구매율 0.47%의 불균형 데이터라 Accuracy 대신 PR-AUC · 상위 10% 지표 사용

def evaluate(y_true, proba, name):
    y_true = np.asarray(y_true)
    # 동점 점수가 있어도 정확히 상위 10%만 선택 (순위 기준)
    k = int(np.ceil(len(proba) * TOP_PCT))
    top = np.zeros(len(proba), dtype=int)
    top[np.argsort(-proba, kind="stable")[:k]] = 1
    base_rate = y_true.mean()
    top_rate = y_true[top == 1].mean()
    return {
        "model": name,
        "PR-AUC": average_precision_score(y_true, proba),
        "PR-AUC baseline": base_rate,
        "ROC-AUC": roc_auc_score(y_true, proba),
        f"Precision@top{int(TOP_PCT*100)}%": precision_score(y_true, top),
        f"Recall(Capture)@top{int(TOP_PCT*100)}%": recall_score(y_true, top),
        f"Lift@top{int(TOP_PCT*100)}%": top_rate / base_rate,
        f"MCC@top{int(TOP_PCT*100)}%": matthews_corrcoef(y_true, top),
    }

In [6]:
# 모델 학습 — Logistic Regression(기준) vs LightGBM

def fit_logistic(X_tr, y_tr):
    pre = ColumnTransformer([
        ("num", Pipeline([("log", FunctionTransformer(np.log1p)),
                          ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ])
    model = Pipeline([
        ("pre", pre),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=2000)),
    ])
    return model.fit(X_tr, y_tr)


def fit_lgbm(X_tr, y_tr):
    # 학습 데이터 안에서 검증셋을 다시 떼어 early stopping (테스트셋 미사용)
    X_fit, X_val, y_fit, y_val = train_test_split(
        X_tr, y_tr, test_size=0.2, stratify=y_tr, random_state=SEED)
    model = lgb.LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.02,
        num_leaves=15,
        min_child_samples=50,
        subsample=0.8, subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=SEED,
        verbose=-1,
    )
    model.fit(X_fit, y_fit,
              eval_set=[(X_val, y_val)],
              eval_metric="binary_logloss",
              callbacks=[lgb.early_stopping(300, first_metric_only=True, verbose=False)])
    print(f"  LightGBM best_iteration : {model.best_iteration_}")
    return model

In [7]:
# SHAP — 중요도, 값별 평균 SHAP(처음 양수가 되는 값 = 데이터 기반 N), 그래프 저장

def compute_shap(model, X_test):
    X_s = X_test.sample(min(SHAP_SAMPLE, len(X_test)), random_state=SEED)
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_s)
    if isinstance(sv, list):          # 구버전 shap : [class0, class1]
        sv = sv[1]
    return sv, X_s


def shap_importance(sv, X_s):
    imp = (pd.DataFrame({"feature": X_s.columns,
                         "mean_abs_shap": np.abs(sv).mean(axis=0)})
             .sort_values("mean_abs_shap", ascending=False)
             .reset_index(drop=True))
    imp["rank"] = imp.index + 1
    imp["group"] = imp["feature"].map(FEATURE_GROUP)
    return imp


def shap_threshold(sv, X_s, feature, max_value=15):
    """값별 평균 SHAP → 기여가 처음 양(+)으로 바뀌는 지점 = 데이터 기반 N"""
    idx = list(X_s.columns).index(feature)
    tbl = (pd.DataFrame({"value": X_s[feature].clip(upper=max_value),
                         "shap": sv[:, idx]})
             .groupby("value")
             .agg(mean_shap=("shap", "mean"), users=("shap", "size"))
             .reset_index())
    positive = tbl[tbl["mean_shap"] > 0]
    n_star = positive["value"].min() if len(positive) else None
    return tbl, n_star


def save_shap_plots(sv, X_s, key):
    shap.summary_plot(sv, X_s, plot_type="bar", show=False, max_display=15)
    plt.title(f"Mean |SHAP| — D7 {key}")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/shap_bar_{key}.png", bbox_inches="tight")
    plt.close()

    shap.summary_plot(sv, X_s, show=False, max_display=15)
    plt.title(f"SHAP summary — D7 {key}")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/shap_beeswarm_{key}.png", bbox_inches="tight")
    plt.close()

    for feat in ["search_then_view_count", "other_view_count"]:
        shap.dependence_plot(feat, sv, X_s, interaction_index=None,
                             xmax="percentile(99)", show=False)
        plt.axhline(0, color="gray", lw=0.8, ls="--")
        plt.title(f"{feat} — D7 {key}")
        plt.tight_layout()
        plt.savefig(f"{OUT_DIR}/shap_dependence_{feat}_{key}.png", bbox_inches="tight")
        plt.close()

In [8]:
# 타깃별 실행

def run_target(df, key, target):
    print(f"\n{'='*70}\n[{key.upper()}] target = {target}  "
          f"| 양성 비율 {df[target].mean():.2%} ({df[target].sum():,}명)\n{'='*70}")

    X, y = df[FEATURES], df[target]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED)

    logit = fit_logistic(X_tr, y_tr)
    lgbm = fit_lgbm(X_tr, y_tr)

    metrics = pd.DataFrame([
        evaluate(y_te, logit.predict_proba(X_te)[:, 1], "Logistic"),
        evaluate(y_te, lgbm.predict_proba(X_te)[:, 1], "LightGBM"),
    ]).set_index("model")
    print("\n[성능 비교]")
    print(metrics.round(4).T.to_string())

    sv, X_s = compute_shap(lgbm, X_te)
    imp = shap_importance(sv, X_s)
    print("\n[SHAP 중요도 상위 10]")
    print(imp.head(10).to_string(index=False))

    thresholds = {}
    for feat in ["search_then_view_count", "other_view_count"]:
        tbl, n_star = shap_threshold(sv, X_s, feat)
        thresholds[feat] = n_star
        tbl.to_csv(f"{OUT_DIR}/shap_threshold_{feat}_{key}.csv", index=False)
        print(f"\n[{feat}] SHAP이 처음 양(+)으로 바뀌는 값 : {n_star}")
        print(tbl.head(11).round(4).to_string(index=False))

    save_shap_plots(sv, X_s, key)
    metrics.to_csv(f"{OUT_DIR}/metrics_{key}.csv")
    imp.to_csv(f"{OUT_DIR}/shap_importance_{key}.csv", index=False)
    return metrics, imp, thresholds

## 4. 실행

In [9]:
from google.colab import auth
auth.authenticate_user()

import os
os.environ["GCP_PROJECT_ID"] = "ga4-analysis-504708"

In [10]:
from google.cloud import bigquery
client = bigquery.Client(project="ga4-analysis-504708")
print(client.query("SELECT COUNT(*) AS n FROM `ga4-analysis-504708.ga4_portfolio.modeling_mart`").to_dataframe())

        n
0  238279


In [11]:
# 실행 : 구매 · 재방문 타깃별 학습 → 평가 → SHAP → X 선정표

df = prepare(load_data())
print(f"분석 대상 : {len(df):,}명 (첫 세션 미구매 · D7 관찰 가능 신규 사용자)")

corr_pairs = check_correlation(df)
print("\n[상관 |ρ| ≥ 0.8 변수 쌍] — SHAP 기여도가 나뉠 수 있어 묶어서 해석")
print(corr_pairs.round(3).to_string(index=False) if len(corr_pairs) else "없음")

results = {k: run_target(df, k, t) for k, t in TARGETS.items()}

# 구매 vs 재방문 SHAP 순위 비교 → X 선정표의 기초
comp = (results["purchase"][1][["feature", "group", "mean_abs_shap", "rank"]]
        .rename(columns={"mean_abs_shap": "shap_purchase", "rank": "rank_purchase"})
        .merge(results["revisit"][1][["feature", "mean_abs_shap", "rank"]]
               .rename(columns={"mean_abs_shap": "shap_revisit", "rank": "rank_revisit"}),
               on="feature"))
reach = {f: (df[f] > 0).mean() for f in NUMERIC_FEATURES}
comp["reach_pct"] = comp["feature"].map(reach)     # 도달 범위 (행동 경험자 비율)
comp = comp.sort_values("rank_purchase").reset_index(drop=True)
comp.to_csv(f"{OUT_DIR}/x_selection_table.csv", index=False, encoding="utf-8-sig")

print(f"\n{'='*70}\n[X 선정표] 구매 · 재방문 SHAP 순위 + 도달 범위\n{'='*70}")
print(comp.round(4).to_string(index=False))
print(f"\n결과 파일 저장 위치 : {os.path.abspath(OUT_DIR)}")

분석 대상 : 236,648명 (첫 세션 미구매 · D7 관찰 가능 신규 사용자)

[상관 |ρ| ≥ 0.8 변수 쌍] — SHAP 기여도가 나뉠 수 있어 묶어서 해석
       feature_1            feature_2  spearman
other_view_count distinct_item_viewed     0.818

[PURCHASE] target = target_purchase_d7  | 양성 비율 0.47% (1,122명)
  LightGBM best_iteration : 196

[성능 비교]
model                   Logistic  LightGBM
PR-AUC                    0.0625    0.0684
PR-AUC baseline           0.0047    0.0047
ROC-AUC                   0.8665    0.8726
Precision@top10%          0.0292    0.0321
Recall(Capture)@top10%    0.6161    0.6786
Lift@top10%               6.1607    6.7857
MCC@top10%                0.1186    0.1330

[SHAP 중요도 상위 10]
               feature  mean_abs_shap  rank     group
  view_promotion_count       0.796589     1 진입 경로(통제)
      other_view_count       0.307179     2    행동(레버)
  distinct_item_viewed       0.254368     3    행동(레버)
        traffic_medium       0.024276     4     통제 변수
     add_to_cart_count       0.023968     5    행동(레버)
       device_categ

### 결과 요약 (구매 모델, 테스트셋)

| 지표 | Logistic | LightGBM |
|---|---:|---:|
| PR-AUC (기준선 0.0047) | 0.0625 | **0.0684** |
| ROC-AUC | 0.8665 | **0.8726** |
| 상위 10% Lift | 6.16배 | **6.79배** |
| 상위 10% Capture | 61.6% | **67.9%** |

- SHAP 중요도: 프로모션 노출(0.797) > 비검색 상품 조회(0.307) > 고유 상품 조회 수(0.254) … 검색 후 상품 조회(0.004, 11위)
- **비검색 상품 조회 횟수별 평균 SHAP값(log-odds)**: 0회 -0.18 → 1회 0.70 → 2회 0.90 → 3회 1.04 → 4회 1.15
  → 0회 → 1회 구간에서 증가폭(+0.88)이 가장 큼
- 프로모션 노출은 메인 페이지 방문과 밀접해 개선 레버에서 제외하고, **'첫 세션 상품 조회 1회 이상'을 행동 X 후보로 재설정**
- 이후 `sql/04_page_path_analysis.sql`에서 view_item의 60%가 목록 페이지에서 발생함을 확인해 **'탐색 경유 상세 도달'로 X를 최종 재정의**

### 프로모션 노출 여부와 무관하게 상세 조회가 구매와 연결되는지 확인

In [12]:
df["promo"] = np.where(df["view_promotion_count"] > 0, "노출", "미노출")
df["detail_views"] = pd.cut(df["other_view_count"] + df["search_then_view_count"],
                            bins=[-1, 0, 2, 5, np.inf], labels=["0", "1-2", "3-5", "6+"])

pd.pivot_table(df, values="target_purchase_d7", index="detail_views", columns="promo",
               aggfunc=["mean", "size"], observed=False).round(4)

mean           size        
promo             노출     미노출     노출     미노출
detail_views                               
0             0.0040  0.0004  61362  130189
1-2           0.0161  0.0029  11207   18251
3-5           0.0290  0.0083   5372    3131
6+            0.0642  0.0301   5607    1529

## 5. A/B 테스트 표본 수

- 1차 지표: **목록 방문자의 상세 도달률** (기준 13.89%, 목록 방문자 132,557명)
- 유의수준 5%(양측), 검정력 80%, 사용자 단위 50:50 무작위 배정
- Primary KPI(첫 세션 미구매자 D7 구매전환율 0.47%)를 직접 1차 지표로 쓰면 필요한 표본이 매우 커서, 상세 도달률을 1차 지표로 두고 구매전환율은 함께 관찰

In [13]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

ALPHA, POWER = 0.05, 0.80
LIST_VISIT_SHARE = 132_557 / 236_648   # 첫 세션 미구매 신규 사용자 중 목록 방문 비율 (56.0%)
power_analysis = NormalIndPower()


def n_per_group(base_rate, relative_uplift):
    """두 비율 비교(양측)에서 그룹당 필요한 표본 수"""
    effect = proportion_effectsize(base_rate * (1 + relative_uplift), base_rate)
    n = power_analysis.solve_power(effect_size=effect, alpha=ALPHA, power=POWER,
                                   ratio=1.0, alternative="two-sided")
    return int(np.ceil(n))


BASE_DETAIL_REACH = 0.1389   # 목록 방문자 상세 도달률
BASE_CVR = 1_122 / 236_648   # 첫 세션 미구매자 D7 구매전환율

rows = []
for uplift in [0.10, 0.15, 0.20]:
    n_list = n_per_group(BASE_DETAIL_REACH, uplift)
    rows.append({
        "상대 개선": f"+{int(uplift * 100)}%",
        "목표 상세 도달률": f"{BASE_DETAIL_REACH * (1 + uplift):.2%}",
        "그룹당 목록 방문자": n_list,
        "그룹당 신규 사용자(환산)": int(np.ceil(n_list / LIST_VISIT_SHARE)),
        "[참고] 구매전환율을 1차 지표로 쓸 때 그룹당": n_per_group(BASE_CVR, uplift),
    })
pd.DataFrame(rows)

,상대 개선,목표 상세 도달률,그룹당 목록 방문자,그룹당 신규 사용자(환산),[참고] 구매전환율을 1차 지표로 쓸 때 그룹당
0,+10%,15.28%,10131,18087,345719
1,+15%,15.97%,4589,8193,157191
2,+20%,16.67%,2629,4694,90389


## 6. 기대 비즈니스 효과

- 기준 모집단은 Primary KPI와 같은 **첫 세션 미구매 신규 사용자 236,648명 / 이후 D7 구매자 1,122명(0.47%)**
- 첫 세션 구매자 1,631명은 이미 구매했으므로 상품 카드 개선 효과의 대상이 아님
- 상세 도달률 개선 비율이 D7 구매전환율에도 같은 비율로 반영된다고 **가정한 시뮬레이션**
- 구매자당 매출은 `sql/03_modeling_mart.sql`의 (B) 결과(미구매자 기준)로 교체하면 더 정확함

In [14]:
BASE_USERS = 236_648          # 첫 세션 미구매 · D7 관찰 가능 신규 사용자
BASE_BUYERS = 1_122           # 이후 D7 구매자
REVENUE_PER_BUYER = 76.89     # 구매자당 D7 평균 매출 (USD) — 03번 (A) 결과
TOTAL_D7_REVENUE = 211_762    # 전체 신규 사용자 D7 매출 (USD)

base_cvr = BASE_BUYERS / BASE_USERS
rows = []
for uplift in [0.10, 0.15, 0.20]:
    extra = BASE_BUYERS * uplift
    rows.append({
        "시나리오": f"상대적 +{int(uplift * 100)}%",
        "예상 D7 구매전환율": f"{base_cvr * (1 + uplift):.2%}",
        "예상 구매자": round(BASE_BUYERS + extra),
        "추정 추가 구매자": round(extra),
        "예상 추가 매출($)": round(extra * REVENUE_PER_BUYER),
        "예상 총 D7 매출($)": round(TOTAL_D7_REVENUE + extra * REVENUE_PER_BUYER),
    })
print(f"기준 D7 구매전환율: {base_cvr:.2%}")
pd.DataFrame(rows)

기준 D7 구매전환율: 0.47%


,시나리오,예상 D7 구매전환율,예상 구매자,추정 추가 구매자,예상 추가 매출($),예상 총 D7 매출($)
0,상대적 +10%,0.52%,1234,112,8627,220389
1,상대적 +15%,0.55%,1290,168,12941,224703
2,상대적 +20%,0.57%,1346,224,17254,229016


## 7. 한계

- 공개 샘플(난독화) 데이터로, 실제 기업의 개선 효과를 입증한 것은 아님
- 첫 세션 행동과 구매의 관계는 연관성이며 인과관계가 아님 → A/B 테스트로 검증 필요
- 모델은 무작위 분할로 평가했으며 시간 순서 기반 검증은 수행하지 않음
- 기대 효과는 가정 기반 시뮬레이션이며 개발·운영 비용은 반영하지 않음